# 01. Tiền xử lý dữ liệu

Notebook này thực hiện quá trình đọc dữ liệu gốc `Online Retail.xlsx`, làm sạch và chuyển đổi sang định dạng giỏ hàng (baskets) phục vụ cho thuật toán ECLAT

In [1]:
# Import các thư viện cần thiết
import pandas as pd
import os

## 1. Đọc dữ liệu raw


In [2]:
# Cài đặt thư mục làm việc về thư mục gốc nếu đang chạy trực tiếp từ thư mục notebooks/
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')

raw_data_path = 'data/raw/Online Retail.xlsx'
print(f"Đang đọc dữ liệu từ: {raw_data_path} ...")

df = pd.read_excel(raw_data_path)
df.head()

Đang đọc dữ liệu từ: data/raw/Online Retail.xlsx ...


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


## 2. Làm sạch dữ liệu
- Loại bỏ khoảng trắng bị dư thừa ở cột `Description`
- Loại bỏ các hàng bị thiếu mã hoá đơn (`InvoiceNo` rỗng)
- Bỏ các hoá đơn bị huỷ (có chứa chữ 'C' trong mã InvoiceNo)
- Chỉ xem xét các giao dịch có số lượng mua `Quantity > 0`
- Bỏ các mặt hàng không có tên `Description`

In [3]:
# Xoá khoảng trắng trùng lặp ở Description
df['Description'] = df['Description'].str.strip()

# Loại bỏ các hàng bị thiếu InvoiceNo
df.dropna(axis=0, subset=['InvoiceNo'], inplace=True)
df['InvoiceNo'] = df['InvoiceNo'].astype('str')

# Loại bỏ các giao dịch bị hủy (InvoiceNo chứa 'C')
df = df[~df['InvoiceNo'].str.contains('C')]

# Loại bỏ các giao dịch có Quantity <= 0
df = df[df['Quantity'] > 0]

# Loại bỏ các hàng thiếu Description
df.dropna(axis=0, subset=['Description'], inplace=True)

print(f"Số lượng bản ghi giao dịch sau khi làm sạch: {df.shape[0]}")

Số lượng bản ghi giao dịch sau khi làm sạch: 530693


## 3. Lưu lại dữ liệu giao dịch sạch
Sẵn sàng phục vụ cho các thuật toán khác về sau.

In [4]:
processed_transactions_path = 'data/processed/transactions_cleaned.csv'
df.to_csv(processed_transactions_path, index=False)
print(f"Đã lưu bảng giao dịch sạch thành công tại: {processed_transactions_path}")
df.head()

Đã lưu bảng giao dịch sạch thành công tại: data/processed/transactions_cleaned.csv


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


## 4. Chuyển đổi định dạng giỏ hàng (Baskets)
Gom các item được mua chung vào 1 danh sách phân tách nhau bằng dấu phẩy theo yêu cầu chuẩn bị dữ liệu thuật toán ECLAT.

In [5]:
# Nhóm theo giao dịch để tạo giỏ hàng cho từng hóa đơn
baskets = df.groupby('InvoiceNo')['Description'].apply(list).reset_index()

# Chuyển list thành chuỗi các mặt hàng cách nhau bởi dấu phẩy
baskets['Items'] = baskets['Description'].apply(lambda x: ','.join(x))
baskets.drop('Description', axis=1, inplace=True)

baskets_path = 'data/processed/baskets.csv'
baskets.to_csv(baskets_path, index=False)

print(f"Đã lưu file giỏ hàng thành công tại: {baskets_path}")
baskets.head()

Đã lưu file giỏ hàng thành công tại: data/processed/baskets.csv


,InvoiceNo,Items
0,536365,"WHITE HANGING HEART T-LIGHT HOLDER,WHITE METAL..."
1,536366,"HAND WARMER UNION JACK,HAND WARMER RED POLKA DOT"
2,536367,"ASSORTED COLOUR BIRD ORNAMENT,POPPY'S PLAYHOUS..."
3,536368,"JAM MAKING SET WITH JARS,RED COAT RACK PARIS F..."
4,536369,BATH BUILDING BLOCK WORD
